# Find the Order - minimal Kaggle starter

This intentionally weak baseline only demonstrates how to read the test pairs and create a valid submission. It predicts `before = 0` for every pair. Replace this constant with your model predictions.

In [1]:
from pathlib import Path
import pandas as pd

INPUT = Path('/kaggle/input/competitions/find-the-order-ioai-2026-unofficial-mirror/find_the_order_test')
pairs = pd.read_csv(INPUT / 'test_pairs.csv')
sample = pd.read_csv(INPUT / 'sample_submission.csv')
print(f'{len(pairs)} test pairs')
display(pairs.head())
display(sample.head())

11730 test pairs


,pair_id,dialogue_id,chunk_i,chunk_j
0,2_0_1,2,0,1
1,2_0_2,2,0,2
2,2_0_3,2,0,3
3,2_0_4,2,0,4
4,2_0_5,2,0,5


,pair_id,before
0,2_0_1,0
1,2_0_2,0
2,2_0_3,0
3,2_0_4,0
4,2_0_5,0


In [4]:
import os
import torch
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model, Wav2Vec2Processor

metadata_path = "/kaggle/input/competitions/find-the-order-ioai-2026-unofficial-mirror/find_the_order_train/train_metadata.csv"
train_dir = "/kaggle/input/competitions/find-the-order-ioai-2026-unofficial-mirror/find_the_order_train/train"

df = pd.read_csv(metadata_path)
df["dialogue_id"] = df["dialogue_id"].astype(int)
df["chunk_id"] = df["chunk_id"].astype(int)

dialogue_counts = df.groupby("dialogue_id")["chunk_id"].transform("count")
df["target"] = df["position"] / dialogue_counts

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "facebook/wav2vec2-base-960h"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name).to(device)
model.eval()

class AudioDataset(Dataset):
    def __init__(self, dataframe, base_dir):
        self.df = dataframe.reset_index(drop=True)
        self.base_dir = base_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        d_id = str(int(row["dialogue_id"]))
        c_id = str(int(row["chunk_id"]))
        
        path = os.path.join(self.base_dir, d_id, f"{c_id}.wav")
        if not os.path.exists(path):
            path = os.path.join(self.base_dir, d_id, f"chunk_{c_id}.wav")

        audio, _ = librosa.load(path, sr=16000)
        return audio, row["target"]

def collate_fn(batch):
    audios = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    inputs = processor(audios, sampling_rate=16000, return_tensors="pt", padding=True)
    return inputs, torch.tensor(targets, dtype=torch.float32)

dataset = AudioDataset(df, train_dir)
dataloader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=2, collate_fn=collate_fn)

X_list = []
y_list = []

with torch.no_grad():
    for inputs, targets in tqdm(dataloader):
        input_values = inputs.input_values.to(device)
        attention_mask = inputs.attention_mask.to(device) if "attention_mask" in inputs else None

        outputs = model(input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state

        if attention_mask is not None:
            mask = model._get_feature_vector_attention_mask(hidden_states.shape[1], attention_mask)
            mask = mask.unsqueeze(-1)
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = hidden_states.mean(dim=1)

        X_list.append(pooled.cpu().numpy())
        y_list.append(targets.numpy())

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 1739/1739 [06:45<00:00,  4.29it/s]


In [18]:
from xgboost import XGBRegressor
import os
import torch
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.linear_model import Ridge, LinearRegression, Lasso

ridge_model = XGBRegressor()
ridge_model.fit(X, y)



XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [6]:
test_dir = "/kaggle/input/competitions/find-the-order-ioai-2026-unofficial-mirror/find_the_order_test/test"
test_meta_path = "/kaggle/input/competitions/find-the-order-ioai-2026-unofficial-mirror/find_the_order_test/test_metadata.csv"

if not os.path.exists(test_meta_path):
    candidates = [
        "/kaggle/input/competitions/find-the-order-ioai-2026-unofficial-mirror/test_metadata.csv",
        "/kaggle/input/competitions/find-the-order-ioai-2026-unofficial-mirror/find_the_order_test/test_pairs.csv",
        "/kaggle/input/competitions/find-the-order-ioai-2026-unofficial-mirror/test.csv"
    ]
    for c in candidates:
        if os.path.exists(c):
            test_meta_path = c
            break

test_pairs_df = pd.read_csv(test_meta_path)
for col in ["dialogue_id", "chunk_i", "chunk_j"]:
    if col in test_pairs_df.columns:
        test_pairs_df[col] = test_pairs_df[col].astype(int)

first_chunk_dict = {}
if "first_chunk" in test_pairs_df.columns:
    first_chunk_dict = test_pairs_df.groupby("dialogue_id")["first_chunk"].first().to_dict()

all_dialogues = test_pairs_df["dialogue_id"].unique()
embeddings = {}

for d_id in tqdm(all_dialogues):
    d_folder = os.path.join(test_dir, str(d_id))
    if not os.path.isdir(d_folder):
        continue
    
    wav_files = [f for f in os.listdir(d_folder) if f.endswith(".wav")]
    for f in wav_files:
        name = f.replace(".wav", "").replace("chunk_", "")
        try:
            c_id = int(name)
        except ValueError:
            continue
            
        wav_path = os.path.join(d_folder, f)
        audio, _ = librosa.load(wav_path, sr=16000)
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_values.to(device)
        
        with torch.no_grad():
            hidden = model(inputs).last_hidden_state
            pooled = hidden.mean(dim=1).squeeze(0).cpu().numpy()
            
        embeddings[(d_id, c_id)] = pooled



100%|██████████| 200/200 [01:03<00:00,  3.16it/s]


In [19]:
import json
import os

test_prefix_path = "/kaggle/input/competitions/find-the-order-ioai-2026-unofficial-mirror/find_the_order_test/test/prefix.json"
with open(test_prefix_path, 'r') as f:
    prefix_dict = json.load(f)

preds_dict = {}
for (d_id, c_id), emb in embeddings.items():
    preds_dict[(d_id, c_id)] = ridge_model.predict(emb.reshape(1, -1))[0]

for d_id in all_dialogues:
    prefixes = prefix_dict.get(str(d_id), [])
    
    if len(prefixes) >= 1:
        first_chunk = prefixes[0]
        if (d_id, first_chunk) in preds_dict:
            preds_dict[(d_id, first_chunk)] = -2.0  
            
    if len(prefixes) >= 2:
        second_chunk = prefixes[1]
        if (d_id, second_chunk) in preds_dict:
            preds_dict[(d_id, second_chunk)] = -1.0 

before_results = []
for _, row in test_pairs_df.iterrows():
    d_id = row["dialogue_id"]
    c_i = row["chunk_i"]
    c_j = row["chunk_j"]
    
    score_i = preds_dict.get((d_id, c_i), 0.0)
    score_j = preds_dict.get((d_id, c_j), 0.0)
    
    before_results.append(1 if score_i < score_j else 0)

sub = pd.DataFrame({
    "pair_id": test_pairs_df["pair_id"],
    "before": before_results
})

sub.to_csv("submission.csv", index=False)

In [10]:
sub

,pair_id,before
0,2_0_1,0
1,2_0_2,0
2,2_0_3,0
3,2_0_4,0
4,2_0_5,0
...,...,...
11725,2415_3_5,0
11726,2415_3_6,1
11727,2415_4_5,0
11728,2415_4_6,1
